# NB83 — The Solenoid Lagrangian

**Target**: Derive the variational principle for the (2,3,5,7)-solenoid in natural (t, R) coordinates. Connect the solenoid metric (NB82) to the cascade ODE (NB79–81) through a Lagrangian framework.

**Key question**: The cascade ODE is first-order dissipative (dR_k/dt + κR_k = f_k). How does this relate to the Riemannian metric g on configuration space?

**Answer**: The metric g defines the **inertial** (second-order) dynamics. The cascade ODE is the **overdamped limit** — when inertia is negligible compared to damping, the second-order E-L equations reduce to first-order gradient flow on the metric. The metric g still governs the geometry of this flow.

**Identity targets**: #187+

**Builds on**: NB43 (W, K, eigenfrequencies), NB82 (metric g, Jacobian M, Schur complement), NB79–81 (cascade ODE)

In [2]:
# ── Setup ──
import sys, numpy as np
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import (SA, RHO, KAPPA, EPSILON, OMEGA,
                               X4, X3, X2, LAM7, X4_LEP,
                               P, PHI, GROUP_EXPONENT,
                               PHYSICAL_CROSSINGS, CP_PAIRS, SM_TARGETS)
from solenoid_system import SolenoidSystem

primes = list(SA.primes)  # [2, 3, 5, 7]
P4 = P                    # 210
n = len(primes)            # 4

# Primorials: P_0=1, P_1=2, P_2=6, P_3=30, P_4=210
PRIMORIALS = [1]
for p in primes:
    PRIMORIALS.append(PRIMORIALS[-1] * p)

# Build W, K from NB43/NB82 (exact Fractions for 5x5 matrices)
def frac_zeros(n):
    A = np.empty((n, n), dtype=object)
    for i in range(n):
        for j in range(n):
            A[i, j] = Fraction(0)
    return A

W = frac_zeros(5)
for i in range(5):
    W[i, i] = Fraction(PRIMORIALS[i])

K = frac_zeros(5)
K[0, 0] = Fraction(1)
for k in range(1, 5):
    p = primes[k - 1]
    K[k, k] = Fraction(p * p + (1 if k < 4 else 0))
    K[k - 1, k] = Fraction(-p)
    K[k, k - 1] = Fraction(-p)

# Build Jacobian M = dtheta/d(t,R) via recursion
# theta_0 = omega*t  (omega = 1 in our units)
# theta_k = (R_{k-1} + theta_{k-1}) / p_k
# So: dtheta_k/dR_{k-1} = 1/p_k,  dtheta_k/dR_j = (dtheta_{k-1}/dR_j)/p_k
M = frac_zeros(5)
M[0, 0] = Fraction(1)  # dtheta_0/dt = omega = 1
for k in range(1, 5):
    pk = primes[k - 1]
    M[k, 0] = M[k - 1, 0] / Fraction(pk)       # time column
    M[k, k] = Fraction(1, pk)                    # diagonal: 1/p_k
    for j in range(1, k):                        # sub-diagonal
        M[k, j] = M[k - 1, j] / Fraction(pk)

print("Jacobian M:")
for i in range(5):
    print("  ", [str(M[i, j]) for j in range(5)])

det_M = Fraction(1)
for i in range(5):
    det_M *= M[i, i]
print(f"\ndet(M) = {det_M}  (expect 1/{P4} = {Fraction(1, P4)})")

# Matrix helpers
def mat_mul(A, B, n=5):
    C = np.empty((n, n), dtype=object)
    for i in range(n):
        for j in range(n):
            C[i, j] = sum(A[i, kk] * B[kk, j] for kk in range(n))
    return C

def mat_T(A, n=5):
    return np.array([[A[j, i] for j in range(n)] for i in range(n)], dtype=object)

MT = mat_T(M)
g = mat_mul(MT, mat_mul(W, M))
K_tR = mat_mul(MT, mat_mul(K, M))

print("\nMetric g_(t,R):")
for i in range(5):
    print("  ", [str(g[i, j]) for j in range(5)])

print("\nStiffness K_(t,R):")
for i in range(5):
    print("  ", [str(K_tR[i, j]) for j in range(5)])

print("\n✓ Setup: W, K, M, g, K_tR (exact Fractions)")

Jacobian M:
   ['1', '0', '0', '0', '0']
   ['1/2', '1/2', '0', '0', '0']
   ['1/6', '1/6', '1/3', '0', '0']
   ['1/30', '1/30', '1/15', '1/5', '0']
   ['1/210', '1/210', '1/105', '1/35', '1/7']

det(M) = 1/210  (expect 1/210 = 1/210)

Metric g_(t,R):
   ['179/105', '74/105', '43/105', '8/35', '1/7']
   ['74/105', '74/105', '43/105', '8/35', '1/7']
   ['43/105', '43/105', '86/105', '16/35', '2/7']
   ['8/35', '8/35', '16/35', '48/35', '6/7']
   ['1/7', '1/7', '2/7', '6/7', '30/7']

Stiffness K_(t,R):
   ['0', '0', '0', '0', '0']
   ['0', '1', '0', '0', '0']
   ['0', '0', '1', '0', '0']
   ['0', '0', '0', '1', '0']
   ['0', '0', '0', '0', '1']

✓ Setup: W, K, M, g, K_tR (exact Fractions)


## §1 — The Quadratic Lagrangian in (t, R) Coordinates

In θ-space, NB43 established the harmonic Lagrangian:

$$\mathcal{L}_{\text{harm}} = \frac{1}{2} \dot{\boldsymbol{\theta}}^\top W \dot{\boldsymbol{\theta}} - \frac{1}{2} \boldsymbol{\theta}^\top K \boldsymbol{\theta}$$

Transform to (t, R) coordinates via θ = M·x where x = (t, R₁, R₂, R₃, R₄):

$$\mathcal{L} = \frac{1}{2} \dot{\mathbf{x}}^\top g \, \dot{\mathbf{x}} - V(\mathbf{x})$$

where g = MᵀWM is the solenoid metric (NB82) and V = ½ xᵀ K_{(t,R)} x = ½ Σ R_k².

**The potential is just ½ Σ R_k²** — unit stiffness in all spatial directions, zero potential for time. This is as simple as a potential can be. All the covering-constraint complexity lives in the **metric**, not the potential.

In [3]:
# ── §1: The quadratic Lagrangian ──
# L = (1/2) x_dot^T g x_dot - (1/2) x^T K_tR x
# Since K_tR = diag(0, 1, 1, 1, 1), V = (1/2)(R_1^2 + R_2^2 + R_3^2 + R_4^2)

print("QUADRATIC LAGRANGIAN IN (t,R) COORDINATES")
print("=" * 65)
print()
print("L = (1/2) x_dot^T g x_dot - V(x)")
print()
print("where:")
print("  x = (t, R_1, R_2, R_3, R_4)")
print("  g = M^T W M  (solenoid metric, NB82)")
print()
print("Potential: V = (1/2) Sum_k R_k^2")
print("  (unit stiffness — all complexity in the metric)")
print()

# Verify the potential: K_tR should be diag(0,1,1,1,1)
K_diag = [K_tR[i, i] for i in range(5)]
K_offdiag = [K_tR[i, j] for i in range(5) for j in range(5) if i != j]
assert K_diag == [Fraction(0), Fraction(1), Fraction(1), Fraction(1), Fraction(1)]
assert all(x == Fraction(0) for x in K_offdiag)
print("✓ K_(t,R) = diag(0, 1, 1, 1, 1) confirmed")

# The Lagrangian can be split: L = T_t + T_mixed + T_R - V
# T_t = (1/2) g_tt * t_dot^2
# T_mixed = g_tR . R_dot * t_dot
# T_R = (1/2) R_dot^T g_RR R_dot
# V = (1/2) R^T R

g_tt = g[0, 0]
g_tR = np.array([g[0, j] for j in range(1, 5)], dtype=object)  # 4-vector
g_RR = np.array([[g[i, j] for j in range(1, 5)] for i in range(1, 5)], dtype=object)  # 4x4

print(f"\n  g_tt = {g_tt}")
print(f"  g_tR = {[str(x) for x in g_tR]}")
print(f"\n  g_RR:")
for i in range(4):
    print(f"    {[str(g_RR[i, j]) for j in range(4)]}")

# Key: g_tt = 179/105 = the zero-mode inertia g_1D from NB82 (#181)
assert g_tt == Fraction(179, 105)
print(f"\n✓ g_tt = 179/105 = g_1D (NB82 Identity #181)")

QUADRATIC LAGRANGIAN IN (t,R) COORDINATES

L = (1/2) x_dot^T g x_dot - V(x)

where:
  x = (t, R_1, R_2, R_3, R_4)
  g = M^T W M  (solenoid metric, NB82)

Potential: V = (1/2) Sum_k R_k^2
  (unit stiffness — all complexity in the metric)

✓ K_(t,R) = diag(0, 1, 1, 1, 1) confirmed

  g_tt = 179/105
  g_tR = ['74/105', '43/105', '8/35', '1/7']

  g_RR:
    ['74/105', '43/105', '8/35', '1/7']
    ['43/105', '86/105', '16/35', '2/7']
    ['8/35', '16/35', '48/35', '6/7']
    ['1/7', '2/7', '6/7', '30/7']

✓ g_tt = 179/105 = g_1D (NB82 Identity #181)


## §2 — Euler-Lagrange Equations

The Euler-Lagrange equations for L = ½ẋᵀgẋ − V(x) with **constant** metric g are:

$$g \ddot{\mathbf{x}} = -\nabla V(\mathbf{x})$$

Since V = ½ Σ R_k² and g is constant:

- **Time**: g_tt ẗ + Σ_k g_{t,k} R̈_k = 0  (no potential for t)
- **Spatial**: Σ_j g_{kj} R̈_j + g_{kt} ẗ = −R_k  (unit stiffness)

These are **second-order** equations. The cascade ODE from NB79–81 is **first-order**:

$$\dot{R}_k + \kappa R_k = f_k(t; \text{lower levels})$$

**The bridge**: The cascade ODE is not the Euler-Lagrange equation. It is the **overdamped limit** — when the inertial term g·ẍ is negligible compared to the damping term κẋ. This is the standard physics of first-order vs second-order dynamics.

But there's a deeper connection: the metric g defines the **geometry** of the flow even in the first-order regime. The natural first-order gradient flow on a Riemannian manifold with metric g is:

$$g \dot{\mathbf{x}} = -\nabla V(\mathbf{x}) + \mathbf{F}_{\text{ext}}$$

This defines the **Riemannian gradient flow**. Let's check whether the cascade ODE can be written in this form.

In [5]:
# ── §2: Euler-Lagrange and the gradient flow ──
# E-L for constant-metric quadratic Lagrangian: g * x_ddot = -grad V
# With V = (1/2) R^T R, grad_R V = R (in the R-block), grad_t V = 0

# The second-order equation in components (4x4 R-block):
# g_RR R_ddot + g_Rt * t_ddot = -R
# g_tt * t_ddot + g_tR . R_ddot = 0

# From the time equation: t_ddot = -(g_tR . R_ddot) / g_tt
# Substitute into spatial:
# (g_RR - g_Rt g_tR / g_tt) R_ddot = -R
# g_schur * R_ddot = -R
# R_ddot = -g_schur^{-1} R

# Compute Schur complement
g_schur = np.empty((4, 4), dtype=object)
for i in range(4):
    for j in range(4):
        g_schur[i, j] = g_RR[i, j] - g_tR[i] * g_tR[j] / g_tt

print("EULER-LAGRANGE EQUATIONS")
print("=" * 65)
print()
print("Second-order EOM: g_schur * R_ddot = -R")
print("where g_schur = g_RR - g_Rt*g_tR / g_tt  (Schur complement)")
print()
print("Schur complement g_schur (exact):")
for i in range(4):
    print("  ", [str(g_schur[i, j]) for j in range(4)])

# Verify det = 180/179 (from NB82 #186)
from sympy import Matrix as SympyMatrix, Rational

g_schur_sym = SympyMatrix(4, 4, [g_schur[i, j] for i in range(4) for j in range(4)])
det_schur = g_schur_sym.det()
print(f"\n  det(g_schur) = {det_schur}  (expect 180/179)")
assert det_schur == Rational(180, 179)
print("  ✓ Matches NB82 Identity #186")

# Eigenfrequencies: eigenvalues of g_schur^{-1}
# Use numerical computation for eigenvalues
g_schur_float = np.array([[float(g_schur[i, j]) for j in range(4)] for i in range(4)])
g_schur_inv_float = np.linalg.inv(g_schur_float)
omega_sq = np.sort(np.linalg.eigvalsh(g_schur_inv_float))

print(f"\n  Eigenvalues of g_schur^{{-1}} (eigenfrequencies omega_k^2):")
for e in omega_sq:
    print(f"    omega^2 = {e:.6f}")

print(f"\n  Sum(omega^2)     = {sum(omega_sq):.6f}  (expect 94/15 = {94/15:.6f})")
print(f"  Product(omega^2) = {np.prod(omega_sq):.6f}  (expect 179/180 = {179/180:.6f})")

# Verify against NB43 values
from scipy.linalg import eigh
W_float = np.diag([float(PRIMORIALS[k]) for k in range(5)])
K_float = np.array([[float(K[i, j]) for j in range(5)] for i in range(5)])

# Symmetric eigenvalue problem: K v = omega^2 W v
# Equivalent to W^{-1/2} K W^{-1/2} u = omega^2 u
Wsqrt_inv = np.diag(1.0 / np.sqrt(np.diag(W_float)))
symK = Wsqrt_inv @ K_float @ Wsqrt_inv
omega_sq_theta = np.sort(np.linalg.eigvalsh(symK))
# Drop the zero eigenvalue (zero mode)
omega_sq_theta_nonzero = omega_sq_theta[omega_sq_theta > 1e-10]

print(f"\n  NB43 theta-space eigenfrequencies (from W^{{-1/2}} K W^{{-1/2}}):")
for e in omega_sq_theta_nonzero:
    print(f"    omega^2 = {e:.6f}")

# Compare
max_diff = max(abs(a - b) for a, b in zip(omega_sq, omega_sq_theta_nonzero))
print(f"\n  Max difference: {max_diff:.2e}")
print(f"  ✓ E-L eigenfrequencies = theta-space eigenfrequencies")

EULER-LAGRANGE EQUATIONS

Second-order EOM: g_schur * R_ddot = -R
where g_schur = g_RR - g_Rt*g_tR / g_tt  (Schur complement)

Schur complement g_schur (exact):
   ['74/179', '43/179', '24/179', '15/179']
   ['43/179', '129/179', '72/179', '45/179']
   ['24/179', '72/179', '240/179', '150/179']
   ['15/179', '45/179', '150/179', '765/179']

  det(g_schur) = 180/179  (expect 180/179)
  ✓ Matches NB82 Identity #186

  Eigenvalues of g_schur^{-1} (eigenfrequencies omega_k^2):
    omega^2 = 0.220621
    omega^2 = 0.748181
    omega^2 = 1.652806
    omega^2 = 3.645058

  Sum(omega^2)     = 6.266667  (expect 94/15 = 6.266667)
  Product(omega^2) = 0.994444  (expect 179/180 = 0.994444)

  NB43 theta-space eigenfrequencies (from W^{-1/2} K W^{-1/2}):
    omega^2 = 0.220621
    omega^2 = 0.748181
    omega^2 = 1.652806
    omega^2 = 3.645058

  Max difference: 8.88e-16
  ✓ E-L eigenfrequencies = theta-space eigenfrequencies


## §3 — The Hamiltonian

The Legendre transform of L = ½ẋᵀgẋ − V(x):

$$\mathbf{p} = \frac{\partial L}{\partial \dot{\mathbf{x}}} = g \dot{\mathbf{x}}$$

$$H = \mathbf{p} \cdot \dot{\mathbf{x}} - L = \frac{1}{2} \mathbf{p}^\top g^{-1} \mathbf{p} + V(\mathbf{x})$$

In the harmonic case V = ½ΣR_k²:

$$H = \frac{1}{2} \mathbf{p}^\top g^{-1} \mathbf{p} + \frac{1}{2} \sum_k R_k^2$$

**The inverse metric g⁻¹** appears in the Hamiltonian. This is the tensor that governs momentum-space dynamics. What does it look like?

In [6]:
# ── §3: The Hamiltonian and inverse metric ──
# H = (1/2) p^T g^{-1} p + (1/2) Sum R_k^2
# Compute g^{-1} exactly

g_sym = SympyMatrix(5, 5, [g[i, j] for i in range(5) for j in range(5)])
g_inv_sym = g_sym.inv()

# Convert back to Fractions
g_inv = np.empty((5, 5), dtype=object)
for i in range(5):
    for j in range(5):
        g_inv[i, j] = Fraction(int(g_inv_sym[i, j].p), int(g_inv_sym[i, j].q))

print("HAMILTONIAN AND INVERSE METRIC")
print("=" * 65)
print()
print("Inverse metric g^{-1}:")
for i in range(5):
    print("  ", [str(g_inv[i, j]) for j in range(5)])

# Check g * g_inv = I
I_check = mat_mul(g, g_inv)
print("\n  g * g^{-1} diagonal:", [str(I_check[i, i]) for i in range(5)])
off_diag_max = max(abs(I_check[i, j]) for i in range(5) for j in range(5) if i != j)
print(f"  Max off-diagonal: {off_diag_max}")

# Structure of g_inv
# Time-time component
print(f"\n  g^{{-1}}_tt = {g_inv[0, 0]}")
print(f"  g^{{-1}}_RR diagonal: {[str(g_inv[i, i]) for i in range(1, 5)]}")

# Check determinant: det(g_inv) = 1/det(g) = 7/(omega^2 * 12) in omega=1 units
# det(g) = 179/105 * 180/179 * product of primorial stuff...
# Actually det(g) = omega^2 * lam/p4 = 1 * 12/7 in omega=1 units
det_g = g_sym.det()
det_g_inv = g_inv_sym.det()
print(f"\n  det(g) = {det_g}  (expect 12/7 in omega=1 units)")
print(f"  det(g^{{-1}}) = {det_g_inv}")
assert det_g == Rational(12, 7)
assert det_g_inv == Rational(7, 12)
print("  ✓ det(g) = 12/7, det(g^{-1}) = 7/12")

# The momentum conjugate to time: p_t = g_{t,mu} x_dot^mu
# For the base solenoid motion: t_dot = 1, R_dot = 0 (zero mode)
# p_t = g_tt = 179/105
# This is the "zero-mode energy" or "total angular momentum" of the solenoid
print(f"\n  Zero-mode momentum: p_t = g_tt * t_dot = {g_tt} (for stationary R)")
print(f"  = g_1D = Sum(1/P_k) = 179/105")
print()

# The Hamiltonian on the zero mode: H = (1/2) g^{-1}_tt p_t^2
# = (1/2) * g^{-1}_tt * g_tt^2 = (1/2) g_tt
H_zero_mode = g_tt / 2
print(f"  H (zero mode) = g_tt / 2 = {H_zero_mode}")
print(f"  = 179/210")

# Fascinating: 179/210 = 179/P_4
assert H_zero_mode == Fraction(179, 210)
print(f"  = 179 / P_4")
print(f"  ✓ The zero-mode energy is 179/P_4 — the prime 179 over the primorial.")

HAMILTONIAN AND INVERSE METRIC

Inverse metric g^{-1}:
   ['1', '-1', '0', '0', '0']
   ['-1', '3', '-1', '0', '0']
   ['0', '-1', '2', '-1/2', '0']
   ['0', '0', '-1/2', '1', '-1/6']
   ['0', '0', '0', '-1/6', '4/15']

  g * g^{-1} diagonal: ['1', '1', '1', '1', '1']
  Max off-diagonal: 0

  g^{-1}_tt = 1
  g^{-1}_RR diagonal: ['3', '2', '1', '4/15']

  det(g) = 12/7  (expect 12/7 in omega=1 units)
  det(g^{-1}) = 7/12
  ✓ det(g) = 12/7, det(g^{-1}) = 7/12

  Zero-mode momentum: p_t = g_tt * t_dot = 179/105 (for stationary R)
  = g_1D = Sum(1/P_k) = 179/105

  H (zero mode) = g_tt / 2 = 179/210
  = 179/210
  = 179 / P_4
  ✓ The zero-mode energy is 179/P_4 — the prime 179 over the primorial.


### Identity #187: The Inverse Metric is Tridiagonal

The inverse metric g⁻¹ is **tridiagonal** — the same sparsity pattern as the original stiffness K! While g itself is dense (lower-triangular Jacobian M fills it), the inverse is sparse:

$$g^{-1} = \begin{pmatrix} 1 & -1 & 0 & 0 & 0 \\ -1 & 3 & -1 & 0 & 0 \\ 0 & -1 & 2 & -\frac{1}{2} & 0 \\ 0 & 0 & -\frac{1}{2} & 1 & -\frac{1}{6} \\ 0 & 0 & 0 & -\frac{1}{6} & \frac{4}{15} \end{pmatrix}$$

Key structural features:
- **g⁻¹_tt = 1** exactly — time has unit inverse inertia
- **Tridiagonal** — nearest-neighbor coupling only (like K, but in momentum space)
- **Off-diagonal entries**: −1, −1, −1/2, −1/6 (sub-diagonal)
- **Diagonal entries**: 1, 3, 2, 1, 4/15

The sub-diagonal pattern −1, −1, −1/2, −1/6 and the diagonal pattern deserve analysis.

In [7]:
# ── Analyze g^{-1} structure ──
print("INVERSE METRIC STRUCTURE ANALYSIS")
print("=" * 65)

# Extract diagonal and sub-diagonal
diag_ginv = [g_inv[i, i] for i in range(5)]
subdiag = [g_inv[i + 1, i] for i in range(4)]

print(f"  Diagonal:      {[str(x) for x in diag_ginv]}")
print(f"  Sub-diagonal:  {[str(x) for x in subdiag]}")

# Check the off-diag pattern: -1, -1, -1/2, -1/6
# The denominators are 1, 1, 2, 6 = P_0, P_0, P_1, P_2
print(f"\n  Sub-diagonal denominators: {[abs(x).denominator for x in subdiag]}")
print(f"  = 1, 1, 2, 6 = P_0, P_0, P_1, P_2")

# Alternative: sub-diagonal[k] = -1/P_{k-1} for k=1,2,3 and -1 for k=0?
# subdiag[0] = -1, subdiag[1] = -1, subdiag[2] = -1/2, subdiag[3] = -1/6
# Pattern: -1/P_0, -1/P_0, -1/P_1, -1/P_2  (shifted primorials)
# Or: all are -1/P_{k-1} if we define P_{-1} = 1

# Check if g^{-1} = W^{-1} K W^{-1} * ... (some scaled form of K?)
# K has off-diag -p_k; g_inv has off-diag -1, -1, -1/2, -1/6
# K_{k,k-1} / P_{k-1} = -p_k/P_{k-1}: -2/1, -3/2, -5/6, -7/30
# Not matching directly

# Let's look at g_inv = (M^T W M)^{-1} = M^{-1} W^{-1} (M^T)^{-1}
# M is lower-triangular with diagonal (1, 1/2, 1/3, 1/5, 1/7)
# M^{-1} also lower-triangular with diagonal (1, 2, 3, 5, 7) = (P_0, p_1, p_2, p_3, p_4)

M_inv_sym = SympyMatrix(5, 5, [M[i, j] for i in range(5) for j in range(5)]).inv()
print(f"\n  M^{{-1}}:")
for i in range(5):
    print("   ", [str(M_inv_sym[i, j]) for j in range(5)])

# W^{-1}
W_inv = np.diag([Fraction(1, PRIMORIALS[k]) for k in range(5)])
print(f"\n  W^{{-1}} = diag({[str(Fraction(1, PRIMORIALS[k])) for k in range(5)]})")

# g^{-1} = M^{-1} W^{-1} (M^T)^{-1}
# The tridiagonal structure of g^{-1} is NOT inherited from K directly.
# It emerges from M^{-1} W^{-1} (M^T)^{-1} where M^{-1} is bidiagonal.

# What IS the structure? Let's look at L = M^{-1}: it's bidiagonal
# L[k,k] = p_k, L[k,k-1] = -p_k (for k >= 1)
# So g^{-1} = L W^{-1} L^T
# g^{-1}[i,j] = Sum_m L[i,m] * (1/P_m) * L[j,m]

print("\n  g^{-1} = L W^{-1} L^T where L = M^{-1} (bidiagonal)")

# Verify: compute L W^{-1} L^T
L = np.empty((5, 5), dtype=object)
for i in range(5):
    for j in range(5):
        L[i, j] = Fraction(int(M_inv_sym[i, j].p), int(M_inv_sym[i, j].q))

g_inv_check = mat_mul(L, mat_mul(W_inv, mat_T(L)))
print("\n  Verification — L W^{-1} L^T:")
for i in range(5):
    print("  ", [str(g_inv_check[i, j]) for j in range(5)])

# Check match
match = all(g_inv_check[i, j] == g_inv[i, j] for i in range(5) for j in range(5))
print(f"\n  Match with g^{{-1}}: {match}")

# The diagonal formula: g^{-1}[k,k] = p_k^2/P_k + p_{k+1}^2/P_{k+1}
# (for boundary terms, the missing term is 0)
# Let's verify
print("\n  Diagonal identity check:")
print(f"  {'k':>2s}  {'g^{-1}[k,k]':>12s}  {'p_k^2/P_k':>12s}  {'p_{k+1}^2/P_{k+1}':>16s}  {'sum':>12s}")
for k in range(5):
    actual = g_inv[k, k]
    if k == 0:
        # L[0,0] = 1, contributes 1^2/P_0 = 1; L[1,0] contributes too
        # g_inv[0,0] = L[0,0]^2/P_0 + L[1,0]^2/P_1
        t1 = L[0, 0]**2 / Fraction(PRIMORIALS[0])
        t2 = L[1, 0]**2 / Fraction(PRIMORIALS[1]) if 1 < 5 else Fraction(0)
        print(f"  {k:2d}  {str(actual):>12s}  L^2/P: {str(t1 + t2):>12s}")
    else:
        # g_inv[k,k] = L[k,k]^2/P_k + L[k+1,k]^2/P_{k+1} (if k+1 < 5)
        t1 = L[k, k]**2 / Fraction(PRIMORIALS[k])
        t2 = L[k + 1, k]**2 / Fraction(PRIMORIALS[k + 1]) if k + 1 < 5 else Fraction(0)
        print(f"  {k:2d}  {str(actual):>12s}  L^2/P: {str(t1 + t2):>12s}")

INVERSE METRIC STRUCTURE ANALYSIS
  Diagonal:      ['1', '3', '2', '1', '4/15']
  Sub-diagonal:  ['-1', '-1', '-1/2', '-1/6']

  Sub-diagonal denominators: [1, 1, 2, 6]
  = 1, 1, 2, 6 = P_0, P_0, P_1, P_2

  M^{-1}:
    ['1', '0', '0', '0', '0']
    ['-1', '2', '0', '0', '0']
    ['0', '-1', '3', '0', '0']
    ['0', '0', '-1', '5', '0']
    ['0', '0', '0', '-1', '7']

  W^{-1} = diag(['1', '1/2', '1/6', '1/30', '1/210'])

  g^{-1} = L W^{-1} L^T where L = M^{-1} (bidiagonal)

  Verification — L W^{-1} L^T:
   ['1', '-1', '0', '0', '0']
   ['-1', '3', '-1', '0', '0']
   ['0', '-1', '2', '-1/2', '0']
   ['0', '0', '-1/2', '1', '-1/6']
   ['0', '0', '0', '-1/6', '4/15']

  Match with g^{-1}: True

  Diagonal identity check:
   k   g^{-1}[k,k]     p_k^2/P_k  p_{k+1}^2/P_{k+1}           sum
   0             1  L^2/P:          3/2
   1             3  L^2/P:         13/6
   2             2  L^2/P:        23/15
   3             1  L^2/P:       88/105
   4          4/15  L^2/P:         7/30


In [10]:
# ── Closed-form formulas for g^{-1} ──
# L = M^{-1} is bidiagonal: L[k,k] = p_k (p_0 := 1), L[k,k-1] = -1
# g^{-1} = L W^{-1} L^T
# Diagonal: g^{-1}[k,k] = sum_m L[k,m]^2 / P_m
#   For k >= 1: terms at m=k-1 and m=k → 1/P_{k-1} + p_k^2/P_k = (1+p_k)/P_{k-1}
#   For k = 0: single term → 1/P_0 = 1

print("CLOSED-FORM FORMULAS FOR g^{-1}")
print("=" * 65)

PRIM = [Fraction(p) for p in PRIMORIALS]
p_ext = [Fraction(1)] + [Fraction(p) for p in primes]  # p_0 = 1

# --- DIAGONAL ---
print("\n  DIAGONAL:")
print("    g^{-1}[0,0] = 1")
print("    g^{-1}[k,k] = (1 + p_k) / P_{k-1}   for k >= 1")
print()
for k in range(5):
    pk = p_ext[k]
    if k == 0:
        formula = Fraction(1)
        label = "1/P_0 = 1"
    else:
        formula = (1 + pk) / PRIM[k - 1]
        label = f"(1 + {pk}) / {PRIMORIALS[k-1]}"
    actual = g_inv[k, k]
    ok = "✓" if formula == actual else "✗"
    print(f"    k={k}: {label} = {formula}  actual = {actual}  {ok}")

# --- SUB-DIAGONAL ---
print(f"\n  SUB-DIAGONAL:")
print(f"    g^{{-1}}[k, k-1] = -p_{{k-1}} / P_{{k-1}}  =  -1 / P_{{k-2}}   for k >= 1")
print()
P_neg1 = Fraction(1)
for k in range(1, 5):
    pk_prev = p_ext[k - 1]
    Pk_prev = PRIM[k - 1]
    formula = -pk_prev / Pk_prev
    actual = g_inv[k, k - 1]
    Pk_prev2 = PRIM[k - 2] if k >= 2 else P_neg1
    alt = Fraction(-1) / Pk_prev2
    ok = "✓" if formula == actual else "✗"
    print(f"    k={k}: -{pk_prev}/{Pk_prev} = {formula} = -1/{Pk_prev2} = {alt}  actual = {actual}  {ok}")

# --- PHYSICAL INTERPRETATION ---
print(f"\n  PHYSICAL INTERPRETATION:")
print(f"    g^{{-1}}_tt = 1: unit inverse inertia for time coordinate")
print(f"    Diagonal decays as ~ 1/P_{{k-1}}: deeper levels have more inertia")
print(f"    Sub-diagonal = -1/P_{{k-2}}: coupling between adjacent levels")
print(f"    Tridiagonal = NEAREST-NEIGHBOR coupling (no long-range leakage)")
print()

trace_ginv = Fraction(1) + sum((1 + p_ext[k]) / PRIM[k - 1] for k in range(1, 5))
det_ginv_exact = Fraction(7, 12)  # from cell 4
det_g_exact = Fraction(12, 7)
print(f"    Tr(g^{{-1}}) = {trace_ginv} = {float(trace_ginv):.6f}")
print(f"    det(g^{{-1}}) = {det_ginv_exact} = {float(det_ginv_exact):.6f}")
print(f"    det(g)      = {det_g_exact} = {float(det_g_exact):.6f}")

# --- sigma_1 observation ---
print(f"\n  NOTE: For primes, sigma_1(p) = 1 + p (sum of divisors)")
print(f"    So g^{{-1}}[k,k] = sigma_1(p_k) / P_{{k-1}} for k >= 1")
print(f"    The inverse metric diagonal is the divisor sum of each prime")
print(f"    divided by the primorial of the previous level.")

CLOSED-FORM FORMULAS FOR g^{-1}

  DIAGONAL:
    g^{-1}[0,0] = 1
    g^{-1}[k,k] = (1 + p_k) / P_{k-1}   for k >= 1

    k=0: 1/P_0 = 1 = 1  actual = 1  ✓
    k=1: (1 + 2) / 1 = 3  actual = 3  ✓
    k=2: (1 + 3) / 2 = 2  actual = 2  ✓
    k=3: (1 + 5) / 6 = 1  actual = 1  ✓
    k=4: (1 + 7) / 30 = 4/15  actual = 4/15  ✓

  SUB-DIAGONAL:
    g^{-1}[k, k-1] = -p_{k-1} / P_{k-1}  =  -1 / P_{k-2}   for k >= 1

    k=1: -1/1 = -1 = -1/1 = -1  actual = -1  ✓
    k=2: -2/2 = -1 = -1/1 = -1  actual = -1  ✓
    k=3: -3/6 = -1/2 = -1/2 = -1/2  actual = -1/2  ✓
    k=4: -5/30 = -1/6 = -1/6 = -1/6  actual = -1/6  ✓

  PHYSICAL INTERPRETATION:
    g^{-1}_tt = 1: unit inverse inertia for time coordinate
    Diagonal decays as ~ 1/P_{k-1}: deeper levels have more inertia
    Sub-diagonal = -1/P_{k-2}: coupling between adjacent levels
    Tridiagonal = NEAREST-NEIGHBOR coupling (no long-range leakage)

    Tr(g^{-1}) = 109/15 = 7.266667
    det(g^{-1}) = 7/12 = 0.583333
    det(g)      = 12/7 = 1.7142

## §4. The Cascade as Driven Resonator

The cascade ODE (NB79–81) is:

$$\dot{R}_k + \kappa R_k = f_k(t;\; \text{lower levels})$$

This is **first-order** — not second-order like the Euler–Lagrange equations. What is the relationship?

**The cascade is a driven, damped oscillator whose resonance structure is determined by the metric.**

Adding Rayleigh dissipation $\mathcal{D} = \frac{\kappa}{2}\dot{R}^\top g_{\text{schur}}\,\dot{R}$ to the E-L equations:

$$g_{\text{schur}}\,\ddot{R} + \kappa\,g_{\text{schur}}\,\dot{R} + R = 0$$

Dividing through:

$$\ddot{R} + \kappa\,\dot{R} + g_{\text{schur}}^{-1}\,R = 0$$

Each eigenmode $r_k$ satisfies $\ddot{r}_k + \kappa\,\dot{r}_k + \omega_k^2\,r_k = 0$ with quality factor $Q_k = \omega_k / \kappa$.

**Key question**: Is the cascade underdamped ($Q > 1$) or overdamped ($Q < 1$)?

### Identity #188: The Schur complement inverse

$g_{\text{schur}}^{-1}$ = lower-right $4 \times 4$ block of $g^{-1}$ (standard block-inversion identity). Since $g^{-1}$ is tridiagonal, $g_{\text{schur}}^{-1}$ inherits the tridiagonal structure with the **same** closed-form entries:

$$g_{\text{schur}}^{-1}[k,k] = \frac{1 + p_{k+1}}{P_k}, \quad g_{\text{schur}}^{-1}[k,k-1] = -\frac{1}{P_{k-1}}$$

In [11]:
# ── §4: Schur complement inverse and Q-factors ──
# g_schur^{-1} = lower-right 4x4 block of g^{-1}
# (standard: if g = [[A, B],[C, D]], S = D-CA^{-1}B, then g^{-1}_{22} = S^{-1})

g_schur_inv = np.array([[g_inv[i+1, j+1] for j in range(4)] for i in range(4)])

print("g_schur^{-1} (from g^{-1} lower-right block):")
for i in range(4):
    print("  ", [str(g_schur_inv[i, j]) for j in range(4)])

# Verify it IS the inverse of g_schur
g_schur_float = np.array([[float(g_schur[i, j]) for j in range(4)] for i in range(4)])
g_sinv_float = np.array([[float(g_schur_inv[i, j]) for j in range(4)] for i in range(4)])
product = g_schur_float @ g_sinv_float
print(f"\ng_schur @ g_schur^{{-1}} = I: {np.allclose(product, np.eye(4))}")
print(f"  max |off-diagonal|: {max(abs(product[i,j]) for i in range(4) for j in range(4) if i != j):.2e}")

# Tridiagonal structure check
is_tridiag = all(g_schur_inv[i, j] == 0 for i in range(4) for j in range(4) if abs(i - j) > 1)
print(f"\ng_schur^{{-1}} is tridiagonal: {is_tridiag}")

# Closed-form verification
# g_schur_inv maps to g^{-1} rows/cols 1..4
# Diagonal: g^{-1}[k,k] = (1+p_k)/P_{k-1}  for k >= 1
# In 4x4 indexing (i = k-1): g_schur_inv[i,i] = (1 + primes[i])/PRIMORIALS[i]
print("\nDiagonal closed-form (from §3 formulas):")
for i in range(4):
    pk = primes[i]
    Pk_prev = PRIMORIALS[i]
    formula = Fraction(1 + pk, Pk_prev)
    actual = g_schur_inv[i, i]
    print(f"  [{i},{i}]: (1+{pk})/{Pk_prev} = {formula}  actual = {actual}  {'✓' if formula == actual else '✗'}")

print("\nSub-diagonal:")
for i in range(1, 4):
    Pk_prev2 = PRIMORIALS[i - 1]
    formula = Fraction(-1, Pk_prev2)
    actual = g_schur_inv[i, i - 1]
    print(f"  [{i},{i-1}]: -1/{Pk_prev2} = {formula}  actual = {actual}  {'✓' if formula == actual else '✗'}")

# ── Q-factors ──
# Eigenfrequencies ω_k² = eigenvalues of g_schur^{-1}
# Already have omega_sq from §2 (eigenvalues of g_schur_inv)
omega_k = np.sqrt(np.sort(np.linalg.eigvalsh(g_sinv_float)))
kappa = float(RHO)  # 1/sqrt(210)

print(f"\n{'='*65}")
print("RESONANCE QUALITY FACTORS")
print(f"{'='*65}")
print(f"\n  Damping rate:     κ = 1/√210 = {kappa:.6f}")
print(f"  Damping time:     τ = 1/κ = √210 = {1/kappa:.4f}")
print()
print(f"  {'Mode':>5s}  {'ω_k':>10s}  {'ω_k²':>10s}  {'T_osc':>10s}  {'Q = ω/κ':>10s}  {'Regime':>12s}")
print(f"  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*12}")
for i, w in enumerate(omega_k):
    Q = w / kappa
    T_osc = 2 * np.pi / w
    regime = "UNDERDAMPED" if Q > 1 else "OVERDAMPED" if Q < 1 else "CRITICAL"
    print(f"  {i:>5d}  {w:>10.6f}  {w**2:>10.6f}  {T_osc:>10.4f}  {Q:>10.4f}  {regime:>12s}")

Q_all = omega_k / kappa
print(f"\n  All Q > 1: {all(Q_all > 1)}")
print(f"  Q range: [{min(Q_all):.2f}, {max(Q_all):.2f}]")
print(f"  Mean Q:  {np.mean(Q_all):.2f}")
print(f"\n  The solenoid is a MODERATE-Q RESONATOR.")
print(f"  All modes are underdamped → resonance peaks are well-defined.")
print(f"  Mass ratios emerge from the DRIVEN RESPONSE at these resonance frequencies.")

# ── Remarkable: product of Q-factors ──
Q_prod = np.prod(Q_all)
Q_prod_exact = np.prod(omega_k) * (1/kappa)**4
# Product of ω_k = sqrt(det(g_schur^{-1})) = sqrt(det_ginv_4x4)
det_sinv = Fraction(1)
for i in range(4):
    for j in range(4):
        pass  # We already know det(g_schur^{-1}) from det(g^{-1})/cofactors
# Use numerical
det_sinv_num = np.linalg.det(g_sinv_float)
omega_prod = np.sqrt(det_sinv_num)
print(f"\n  Product of ω_k = √det(g_schur^{{-1}}) = {omega_prod:.6f}")
print(f"  det(g_schur^{{-1}}) = {det_sinv_num:.6f}")

# Compare with known: det(g_schur) = det_schur from §2
det_schur_num = float(det_schur)
print(f"  det(g_schur) = {float(det_schur):.6f} = {det_schur}")
print(f"  det(g_schur^{{-1}}) = 1/det(g_schur) = {1/det_schur_num:.6f}")
print(f"  Consistent: {abs(det_sinv_num - 1/det_schur_num) < 1e-12}")

g_schur^{-1} (from g^{-1} lower-right block):
   ['3', '-1', '0', '0']
   ['-1', '2', '-1/2', '0']
   ['0', '-1/2', '1', '-1/6']
   ['0', '0', '-1/6', '4/15']

g_schur @ g_schur^{-1} = I: True
  max |off-diagonal|: 1.11e-16

g_schur^{-1} is tridiagonal: True

Diagonal closed-form (from §3 formulas):
  [0,0]: (1+2)/1 = 3  actual = 3  ✓
  [1,1]: (1+3)/2 = 2  actual = 2  ✓
  [2,2]: (1+5)/6 = 1  actual = 1  ✓
  [3,3]: (1+7)/30 = 4/15  actual = 4/15  ✓

Sub-diagonal:
  [1,0]: -1/1 = -1  actual = -1  ✓
  [2,1]: -1/2 = -1/2  actual = -1/2  ✓
  [3,2]: -1/6 = -1/6  actual = -1/6  ✓

RESONANCE QUALITY FACTORS

  Damping rate:     κ = 1/√210 = 0.069007
  Damping time:     τ = 1/κ = √210 = 14.4914

   Mode         ω_k        ω_k²       T_osc     Q = ω/κ        Regime
  ─────  ──────────  ──────────  ──────────  ──────────  ────────────
      0    0.469704    0.220621     13.3769      6.8067   UNDERDAMPED
      1    0.864974    0.748181      7.2640     12.5347   UNDERDAMPED
      2    1.285615    1

### Identity #189: The Cascade Jacobian and the Shared Building Block

The cascade ODE (from `solenoid_system.py`) has the time-averaged Jacobian:

$$J_{\text{avg}} = \kappa \begin{pmatrix} -1 & 0 & 0 & 0 \\ 1/p_1 & -1 & 0 & 0 \\ 0 & 1/p_2 & -1 & 0 \\ 0 & 0 & 1/p_3 & -1 \end{pmatrix}$$

This is **lower bidiagonal** — the same structure as $M^{-1}$ (the inverse covering map).

Meanwhile $g_{\text{schur}}^{-1}$ is **symmetric tridiagonal** — $M^{-1}$ symmetrized through $W^{-1}$.

**Both structures derive from the same building block: $M^{-1}$.**

- The cascade uses $M^{-1}$ **asymmetrically** (covering map flow is directional: outer drives inner)
- The metric uses $M^{-1}$ **symmetrically** through $g^{-1} = M^{-1}\,W^{-1}\,(M^{-1})^\top$
- The metric is the **quadratic form** built from the same linear operator that governs the cascade

In [12]:
# ── Time-averaged cascade Jacobian vs metric ──
from fractions import Fraction

# Cascade Jacobian (time-averaged, linearized around R=0)
# From cascade_ode: J[k,k] = -kappa, J[k,k-1] = kappa/p_{k-1}
# (sin terms average to zero over full solenoid period)
J_avg = np.zeros((4, 4), dtype=object)
for k in range(4):
    J_avg[k, k] = Fraction(-1)  # -kappa (factor kappa out)
    if k > 0:
        J_avg[k, k - 1] = Fraction(1, primes[k - 1])

print("Time-averaged cascade Jacobian / κ:")
for i in range(4):
    print("  ", [str(J_avg[i, j]) for j in range(4)])

# The sub-diagonal entries are 1/p_k:
cascade_subdiag = [Fraction(1, primes[k]) for k in range(3)]
metric_subdiag = [g_schur_inv[i + 1, i] for i in range(3)]

print(f"\nSub-diagonal comparison:")
print(f"  Cascade J_avg/κ:  {[str(s) for s in cascade_subdiag]}")
print(f"  g_schur^{{-1}}:     {[str(s) for s in metric_subdiag]}")

# The building block is M^{-1}: bidiag(diag=[1,p_1,...,p_4], sub=-1)
# Cascade: J_avg/κ = -I + diag(1/p_k, sub)  (M^{-1} structure, column-scaled)
# Metric: g_schur^{-1} = (M^{-1} W^{-1} (M^{-1})^T) restricted to 4x4

# The KEY structural identity: both are NEAREST-NEIGHBOR
# J_avg couples level k only to level k-1 (directional: outer → inner)
# g_schur^{-1} couples level k to k-1 AND k+1 (symmetric: bidirectional)

# Compute: is J_avg + J_avg^T related to g_schur^{-1}?
J_sym = np.zeros((4, 4), dtype=object)
for i in range(4):
    for j in range(4):
        J_sym[i, j] = J_avg[i, j] + J_avg[j, i]  # symmetrize

print(f"\nJ_avg/κ + (J_avg/κ)^T (symmetrized cascade):")
for i in range(4):
    print("  ", [str(J_sym[i, j]) for j in range(4)])

# Compare structure: both are tridiagonal, both use primes
# Cascade: sub-diag = 1/p_k      (individual primes)
# Metric:  sub-diag = -1/P_{k-1} (primorials = products of primes)
# Metric sub-diagonal encodes CUMULATIVE coupling (primorial)
# Cascade sub-diagonal encodes SINGLE-STEP coupling (individual prime)
print(f"\nStructural relationship:")
for k in range(3):
    p_k = primes[k]
    P_k = PRIMORIALS[k]
    ratio = Fraction(1, p_k) / abs(metric_subdiag[k])
    print(f"  Level {k+1}←{k}: cascade 1/{p_k}, metric -1/{P_k}  ratio = {ratio} = P_{k}/{p_k}")

# The ratio is P_{k-1}/1 — the metric accumulates primorial weight
print(f"\n  Cascade coupling = metric coupling × P_{{k-1}} / p_{{k}}")
print(f"  The metric ENCODES the cascade but with primorial normalization.")
print(f"  This is because g^{{-1}} = M^{{-1}} W^{{-1}} (M^{{-1}})^T:")
print(f"    M^{{-1}} provides the 1/p_k coupling (cascade structure)")
print(f"    W^{{-1}} weight 1/P_k provides the primorial normalization")
print(f"    Symmetrization gives the bidirectional metric coupling")

Time-averaged cascade Jacobian / κ:
   ['-1', '0', '0', '0']
   ['1/2', '-1', '0', '0']
   ['0', '1/3', '-1', '0']
   ['0', '0', '1/5', '-1']

Sub-diagonal comparison:
  Cascade J_avg/κ:  ['1/2', '1/3', '1/5']
  g_schur^{-1}:     ['-1', '-1/2', '-1/6']

J_avg/κ + (J_avg/κ)^T (symmetrized cascade):
   ['-2', '1/2', '0', '0']
   ['1/2', '-2', '1/3', '0']
   ['0', '1/3', '-2', '1/5']
   ['0', '0', '1/5', '-2']

Structural relationship:
  Level 1←0: cascade 1/2, metric -1/1  ratio = 1/2 = P_0/2
  Level 2←1: cascade 1/3, metric -1/2  ratio = 2/3 = P_1/3
  Level 3←2: cascade 1/5, metric -1/6  ratio = 6/5 = P_2/5

  Cascade coupling = metric coupling × P_{k-1} / p_{k}
  The metric ENCODES the cascade but with primorial normalization.
  This is because g^{-1} = M^{-1} W^{-1} (M^{-1})^T:
    M^{-1} provides the 1/p_k coupling (cascade structure)
    W^{-1} weight 1/P_k provides the primorial normalization
    Symmetrization gives the bidirectional metric coupling


---

## Scorecard

NB83 establishes four structural identities (#187–#190), building on the metric from NB82:

| # | Identity | Value | Verdict |
|---|----------|-------|---------|
| 187 | g⁻¹ tridiagonal: g⁻¹ = LW⁻¹Lᵀ, L = M⁻¹ bidiag(1,p₁,..,p₄), diag = (1+pₖ)/Pₖ₋₁, sub = -1/Pₖ₋₂ | Exact (Fraction arithmetic) | PASS |
| 188 | g_schur⁻¹ = lower-right 4×4 of g⁻¹; inherits tridiagonal with same closed-form | Verified: g_schur·g_schur⁻¹ = I₄ | PASS |
| 189 | All Q-factors > 1: Q ∈ [6.81, 27.67]; solenoid is underdamped resonator | 4/4 modes underdamped | STRUCTURAL |
| 190 | Cascade Jacobian and metric share M⁻¹ building block (nearest-neighbor coupling) | Cascade = 1/pₖ, metric = 1/Pₖ₋₁ | STRUCTURAL |

In [13]:
# ── NB83 SCORECARD ──
print("NB83 SCORECARD — The Solenoid Lagrangian")
print("=" * 65)
print()
print(f"{'#':>4s}  {'Identity':40s}  {'Verdict':>10s}")
print(f"{'─'*4}  {'─'*40}  {'─'*10}")
print(f" 187  g^{{-1}} tridiagonal = LW^{{-1}}L^T            PASS")
print(f"      diag = (1+p_k)/P_{{k-1}}, sub = -1/P_{{k-2}}")
print(f" 188  g_schur^{{-1}} = 4x4 block of g^{{-1}}        PASS")
print(f"      inherits tridiagonal closed-form")
print(f" 189  All Q > 1: range [{min(Q_all):.2f}, {max(Q_all):.2f}]     STRUCTURAL")
print(f"      solenoid = underdamped resonator")
print(f" 190  Cascade & metric share M^{{-1}} block       STRUCTURAL")
print(f"      nearest-neighbor coupling from primes")
print()
print(f"New identities:     4 (#187–#190)")
print(f"  PASS:             2")
print(f"  STRUCTURAL:       2")
print(f"Running total:    181 predictions/identities, 0 free parameters")

NB83 SCORECARD — The Solenoid Lagrangian

   #  Identity                                     Verdict
────  ────────────────────────────────────────  ──────────
 187  g^{-1} tridiagonal = LW^{-1}L^T            PASS
      diag = (1+p_k)/P_{k-1}, sub = -1/P_{k-2}
 188  g_schur^{-1} = 4x4 block of g^{-1}        PASS
      inherits tridiagonal closed-form
 189  All Q > 1: range [6.81, 27.67]     STRUCTURAL
      solenoid = underdamped resonator
 190  Cascade & metric share M^{-1} block       STRUCTURAL
      nearest-neighbor coupling from primes

New identities:     4 (#187–#190)
  PASS:             2
  STRUCTURAL:       2
Running total:    181 predictions/identities, 0 free parameters
